In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/TGMT/PROJECT/Colab/req23/TransReID-official
!python -m pip install -q -r requirements.txt yacs

/content/drive/.shortcut-targets-by-id/1ibso21ORkUy5SmN88-1AeBQ-46UhPoYh/TGMT/PROJECT/Colab/req23/TransReID-official


In [ ]:
!mkdir -p /content/data
!unzip -q /content/market1501.zip -d /content/data

In [ ]:
!ls /content/data
!ls /content/data/market1501

market1501
bounding_box_test  bounding_box_train  gt_bbox	gt_query  query  readme.txt


In [ ]:
!OUTPUT_DIR="/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/logs/market_vit_transreid_stride_reliability_localjpm_imagenet"; \
mkdir -p "$OUTPUT_DIR"; \
python train.py \
  --config_file configs/Market/vit_transreid_stride_reliability_full_imagenet.yml \
  MODEL.DEVICE_ID "('0')" \
  MODEL.PRETRAIN_CHOICE "imagenet" \
  MODEL.PRETRAIN_PATH "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/checkpoint/jx_vit_base_p16_224-80ecf9dd.pth')" \
  DATASETS.ROOT_DIR "('/content/data')" \
  SOLVER.IMS_PER_BATCH "32" \
  DATALOADER.NUM_WORKERS "2" \
  MODEL.REL_METRIC_LOSS_WEIGHT "0.00" \
  MODEL.REL_CONSIST_LOSS_WEIGHT "0.10" \
  MODEL.REL_VIS_LOSS_WEIGHT "0.10" \
  MODEL.REL_FUSION_ALPHA "0.05" \
  MODEL.REL_GATE_INIT "-4.0" \
  MODEL.REL_REG_WEIGHT "0.0" \
  OUTPUT_DIR "('$OUTPUT_DIR')"


2026-04-15 18:04:38,494 transreid INFO: Saving model in the path :/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/logs/market_vit_transreid_stride_reliability_localjpm_imagenet
2026-04-15 18:04:38,495 transreid INFO: Namespace(config_file='configs/Market/vit_transreid_stride_reliability_full_imagenet.yml', opts=['MODEL.DEVICE_ID', "('0')", 'MODEL.PRETRAIN_CHOICE', 'imagenet', 'MODEL.PRETRAIN_PATH', "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/checkpoint/jx_vit_base_p16_224-80ecf9dd.pth')", 'DATASETS.ROOT_DIR', "('/content/data')", 'SOLVER.IMS_PER_BATCH', '32', 'DATALOADER.NUM_WORKERS', '2', 'MODEL.REL_METRIC_LOSS_WEIGHT', '0.00', 'MODEL.REL_CONSIST_LOSS_WEIGHT', '0.10', 'MODEL.REL_VIS_LOSS_WEIGHT', '0.10', 'MODEL.REL_FUSION_ALPHA', '0.05', 'MODEL.REL_GATE_INIT', '-4.0', 'MODEL.REL_REG_WEIGHT', '0.0', 'OUTPUT_DIR', "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/logs/market_vit_transreid_stride_reliability_localjpm_imagenet')"], local_rank=0)
2026-04-15 18:04:38,496 transreid I

dataset Occluded Duke

In [ ]:
%cd /content
!cp "/content/drive/MyDrive/TGMT/PROJECT/Colab/data/DukeMTMC-reID.zip" /content/
!ls -lh /content/DukeMTMC-reID.zip

/content
-rw------- 1 root root 142M Apr 16 06:47 /content/DukeMTMC-reID.zip


In [ ]:
%cd /content

!unzip -q "/content/DukeMTMC-reID.zip" -d /content/DukeMTMC-reID

!find /content/DukeMTMC-reID -maxdepth 3 -type d | sort

/content
/content/DukeMTMC-reID
/content/DukeMTMC-reID/bounding_box_test
/content/DukeMTMC-reID/bounding_box_train
/content/DukeMTMC-reID/query


In [ ]:
%cd /content
!git clone https://github.com/lightas/Occluded-DukeMTMC-Dataset.git

/content
fatal: destination path 'Occluded-DukeMTMC-Dataset' already exists and is not an empty directory.


In [ ]:
%cd /content
!rm -rf /content/duke_src
!mkdir -p /content/duke_src
!unzip -q "/content/drive/MyDrive/TGMT/PROJECT/Colab/data/DukeMTMC-reID.zip" -d /content/duke_src
!find /content/duke_src -maxdepth 3 -type d | sort

/content
/content/duke_src
/content/duke_src/bounding_box_test
/content/duke_src/bounding_box_train
/content/duke_src/query


In [ ]:
%%writefile /content/convert_occduke_from_folder.py
import os
import shutil

target_dir = '/content/Occluded-DukeMTMC-Dataset/Occluded_Duke'
repo_dir = '/content/Occluded-DukeMTMC-Dataset/Occluded_Duke'

def find_duke_root(base):
    candidates = []
    for root, dirs, files in os.walk(base):
        dset = set(dirs)
        if {'bounding_box_train', 'bounding_box_test', 'query'}.issubset(dset):
            candidates.append(root)
    if not candidates:
        raise RuntimeError("Không tìm thấy thư mục DukeMTMC-reID hợp lệ")
    candidates.sort(key=lambda x: len(x))
    return candidates[0]

def generate_new_split(origin_duke_dir, split, folder_name):
    list_path = os.path.join(repo_dir, f'{split}.list')
    with open(list_path, 'r') as f:
        imgs = [line.strip() for line in f if line.strip()]

    source_split = os.path.join(origin_duke_dir, folder_name)
    target_split = os.path.join(target_dir, folder_name)
    os.makedirs(target_split, exist_ok=True)

    missing = []
    for img in imgs:
        p1 = os.path.join(source_split, img)
        p2 = os.path.join(origin_duke_dir, 'bounding_box_test', img)
        if os.path.isfile(p1):
            src = p1
        elif os.path.isfile(p2):
            src = p2
        else:
            missing.append(img)
            continue
        shutil.copy(src, os.path.join(target_split, img))

    print(f"{folder_name}: copied {len(imgs)-len(missing)}/{len(imgs)}")
    if missing:
        print("Missing sample:", missing[:10])
        raise FileNotFoundError(f"Thiếu {len(missing)} ảnh trong split {folder_name}")

origin_duke_dir = find_duke_root('/content/duke_src')
print("Found Duke root:", origin_duke_dir)

os.makedirs(target_dir, exist_ok=True)
generate_new_split(origin_duke_dir, 'train', 'bounding_box_train')
generate_new_split(origin_duke_dir, 'gallery', 'bounding_box_test')
generate_new_split(origin_duke_dir, 'query', 'query')

print("\nDone:", target_dir)

Writing /content/convert_occduke_from_folder.py


In [ ]:
%cd /content
!git clone https://github.com/lightas/Occluded-DukeMTMC-Dataset.git

/content
fatal: destination path 'Occluded-DukeMTMC-Dataset' already exists and is not an empty directory.


In [ ]:
!python /content/convert_occduke_from_folder.py

Found Duke root: /content/duke_src
bounding_box_train: copied 15618/15618
bounding_box_test: copied 17661/17661
query: copied 2210/2210

Done: /content/Occluded-DukeMTMC-Dataset/Occluded_Duke


In [ ]:
!find /content/Occluded-DukeMTMC-Dataset/Occluded_Duke -maxdepth 2 -type d | sort

/content/Occluded-DukeMTMC-Dataset/Occluded_Duke
/content/Occluded-DukeMTMC-Dataset/Occluded_Duke/bounding_box_test
/content/Occluded-DukeMTMC-Dataset/Occluded_Duke/bounding_box_train
/content/Occluded-DukeMTMC-Dataset/Occluded_Duke/query


In [ ]:
%cd /content/drive/MyDrive/TGMT/PROJECT/Colab/req23/TransReID-official

!OUTPUT_DIR="/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/logs/occduke_vit_transreid_stride_reliability_localjpm_imagenet"; \
mkdir -p "$OUTPUT_DIR"; \
python train.py \
  --config_file configs/Market/vit_transreid_stride_reliability_full_imagenet.yml \
  MODEL.DEVICE_ID "('0')" \
  MODEL.PRETRAIN_CHOICE "imagenet" \
  MODEL.PRETRAIN_PATH "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/checkpoint/jx_vit_base_p16_224-80ecf9dd.pth')" \
  DATASETS.NAMES "'occ_duke'" \
  DATASETS.ROOT_DIR "('/content/Occluded-DukeMTMC-Dataset')" \
  SOLVER.IMS_PER_BATCH "32" \
  DATALOADER.NUM_WORKERS "2" \
  MODEL.REL_METRIC_LOSS_WEIGHT "0.00" \
  MODEL.REL_CONSIST_LOSS_WEIGHT "0.10" \
  MODEL.REL_VIS_LOSS_WEIGHT "0.10" \
  MODEL.REL_FUSION_ALPHA "0.05" \
  MODEL.REL_GATE_INIT "-4.0" \
  MODEL.REL_REG_WEIGHT "0.0" \
  OUTPUT_DIR "('$OUTPUT_DIR')"


/content/drive/.shortcut-targets-by-id/1ibso21ORkUy5SmN88-1AeBQ-46UhPoYh/TGMT/PROJECT/Colab/req23/TransReID-official
Traceback (most recent call last):
  File "/content/drive/.shortcut-targets-by-id/1ibso21ORkUy5SmN88-1AeBQ-46UhPoYh/TGMT/PROJECT/Colab/req23/TransReID-official/train.py", line 9, in <module>
    from datasets import make_dataloader
  File "/content/drive/.shortcut-targets-by-id/1ibso21ORkUy5SmN88-1AeBQ-46UhPoYh/TGMT/PROJECT/Colab/req23/TransReID-official/datasets/__init__.py", line 1, in <module>
    from .make_dataloader import make_dataloader
  File "/content/drive/.shortcut-targets-by-id/1ibso21ORkUy5SmN88-1AeBQ-46UhPoYh/TGMT/PROJECT/Colab/req23/TransReID-official/datasets/make_dataloader.py", line 1, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 430, in <module>
    _load_global_deps()
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 381, in _load_global_deps
    _preload_cuda_deps()
  Fi

In [ ]:
%cd /content/drive/MyDrive/TGMT/PROJECT/Colab/req23/TransReID-official

!OUTPUT_DIR="/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/logs/occduke_vit_transreid_stride_reliability_localjpm_imagenet"; \
mkdir -p "$OUTPUT_DIR"; \
python train.py \
  --config_file configs/Market/vit_transreid_stride_reliability_full_imagenet.yml \
  MODEL.DEVICE_ID "('0')" \
  MODEL.PRETRAIN_CHOICE "imagenet" \
  MODEL.PRETRAIN_PATH "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/checkpoint/jx_vit_base_p16_224-80ecf9dd.pth')" \
  DATASETS.NAMES "'occ_duke'" \
  DATASETS.ROOT_DIR "('/content/Occluded-DukeMTMC-Dataset')" \
  SOLVER.IMS_PER_BATCH "32" \
  DATALOADER.NUM_WORKERS "2" \
  SOLVER.RESUME "True" \
  SOLVER.RESUME_PATH "('$OUTPUT_DIR/latest_checkpoint.pth')" \
  MODEL.REL_METRIC_LOSS_WEIGHT "0.00" \
  MODEL.REL_CONSIST_LOSS_WEIGHT "0.10" \
  MODEL.REL_VIS_LOSS_WEIGHT "0.10" \
  MODEL.REL_FUSION_ALPHA "0.05" \
  MODEL.REL_GATE_INIT "-4.0" \
  MODEL.REL_REG_WEIGHT "0.0" \
  OUTPUT_DIR "('$OUTPUT_DIR')"


/content/drive/.shortcut-targets-by-id/1ibso21ORkUy5SmN88-1AeBQ-46UhPoYh/TGMT/PROJECT/Colab/req23/TransReID-official
2026-04-16 06:49:28,671 transreid INFO: Saving model in the path :/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/logs/occduke_vit_transreid_stride_reliability_localjpm_imagenet
2026-04-16 06:49:28,671 transreid INFO: Namespace(config_file='configs/Market/vit_transreid_stride_reliability_full_imagenet.yml', opts=['MODEL.DEVICE_ID', "('0')", 'MODEL.PRETRAIN_CHOICE', 'imagenet', 'MODEL.PRETRAIN_PATH', "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/checkpoint/jx_vit_base_p16_224-80ecf9dd.pth')", 'DATASETS.NAMES', "'occ_duke'", 'DATASETS.ROOT_DIR', "('/content/Occluded-DukeMTMC-Dataset')", 'SOLVER.IMS_PER_BATCH', '32', 'DATALOADER.NUM_WORKERS', '2', 'SOLVER.RESUME', 'True', 'SOLVER.RESUME_PATH', "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/logs/occduke_vit_transreid_stride_reliability_localjpm_imagenet/latest_checkpoint.pth')", 'MODEL.REL_METRIC_LOSS_WEIGHT', '0.00

In [ ]:
%cd /content/drive/MyDrive/TGMT/PROJECT/Colab/req23/TransReID-official

!python test.py \
  --config_file configs/Market/vit_transreid_stride_reliability_full_imagenet.yml \
  MODEL.DEVICE_ID "('0')" \
  MODEL.PRETRAIN_CHOICE "imagenet" \
  MODEL.PRETRAIN_PATH "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/checkpoint/jx_vit_base_p16_224-80ecf9dd.pth')" \
  DATASETS.NAMES "'occ_duke'" \
  DATASETS.ROOT_DIR "('/content/Occluded-DukeMTMC-Dataset')" \
  TEST.WEIGHT "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/logs/occduke_vit_transreid_stride_reliability_localjpm_imagenet/best_model.pth')"


/content/drive/.shortcut-targets-by-id/1ibso21ORkUy5SmN88-1AeBQ-46UhPoYh/TGMT/PROJECT/Colab/req23/TransReID-official
2026-04-16 08:52:24,145 transreid INFO: Namespace(config_file='configs/Market/vit_transreid_stride_reliability_full_imagenet.yml', opts=['MODEL.DEVICE_ID', "('0')", 'MODEL.PRETRAIN_CHOICE', 'imagenet', 'MODEL.PRETRAIN_PATH', "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/checkpoint/jx_vit_base_p16_224-80ecf9dd.pth')", 'DATASETS.NAMES', "'occ_duke'", 'DATASETS.ROOT_DIR', "('/content/Occluded-DukeMTMC-Dataset')", 'TEST.WEIGHT', "('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/logs/occduke_vit_transreid_stride_reliability_localjpm_imagenet/best_model.pth')"])
2026-04-16 08:52:24,146 transreid INFO: Loaded configuration file configs/Market/vit_transreid_stride_reliability_full_imagenet.yml
2026-04-16 08:52:24,147 transreid INFO: 
MODEL:
  PRETRAIN_CHOICE: 'imagenet'
  PRETRAIN_PATH: '/home/heshuting/.cache/torch/checkpoints/jx_vit_base_p16_224-80ecf9dd.pth'
  METRIC_LO